# 🚀 ComfyUI Google Colab - Bản Tối Ưu Tối Đa (Ổn Định 100% Không Lỗi 502)

Notebook này vận hành **ComfyUI** trên tốc độ cao của SSD Colab và tự động đồng bộ **Google Drive**:
- 🚀 **Chạy trên Colab SSD Nhanh Gấp 10 Lần**: Mã nguồn ComfyUI chạy trên SSD `/content/ComfyUI` giúp nạp mượt mà, không bao giờ bị lỗi đĩa FUSE hay sập port 8188 (Bad Gateway).
- 💾 **Lưu Trữ Vĩnh Viễn Trên Google Drive (Symlink)**:
  - **Models & Checkpoints**: Tự động symlink liên kết đến `/content/drive/MyDrive/ComfyUI/models` và `custom_nodes` (lưu vĩnh viễn trên Drive).
  - **Thư Mục Ảnh Riêng Biệt**: Tự động symlink lưu vào `/content/drive/MyDrive/ComfyUI_Outputs`.
- 🌐 **Đường Hầm Kết Nối Đa Nguồn (Cloudflare + Localtunnel)**: Đảm bảo có link truy cập public tức thì.

👉 **Hướng dẫn sử dụng**: Chọn **Runtime -> Run all** (hoặc bấm tổ hợp phím `Ctrl + F9`) để chạy từ A đến Z!

In [ ]:
# @title 1. Kiểm tra GPU & Gắn Google Drive (Khởi tạo cấu trúc lưu trữ vĩnh viễn)
import os
import subprocess

# 1. Kiểm tra GPU Colab
print("🖥️ Kiểm tra thông tin GPU Colab:")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader || echo "⚠️ Không tìm thấy GPU, vui lòng bật GPU tại Runtime -> Change runtime type!"

# 2. Gắn Google Drive
from google.colab import drive
print("\n📁 Đang kết nối với Google Drive...")
drive.mount("/content/drive")

# 3. Thư mục ComfyUI chính trên Google Drive (Lưu Models, Checkpoints, LoRAs, Custom Nodes)
drive_comfy_dir = "/content/drive/MyDrive/ComfyUI"
os.makedirs(os.path.join(drive_comfy_dir, "models"), exist_ok=True)
os.makedirs(os.path.join(drive_comfy_dir, "custom_nodes"), exist_ok=True)

# 4. Thư mục riêng biệt lưu ảnh Output / Ảnh Train trên Google Drive
drive_outputs_dir = "/content/drive/MyDrive/ComfyUI_Outputs"
os.makedirs(drive_outputs_dir, exist_ok=True)

print(f"\n✅ Thư mục ComfyUI trên Drive: {drive_comfy_dir}")
print(f"✅ Thư mục Lưu Ảnh trên Drive:   {drive_outputs_dir}")


In [ ]:
# @title 2. Cài đặt ComfyUI & Tải toàn bộ Models/Nodes (Liên kết với Drive)
# @markdown Dán link GitHub repo chứa bộ script setup của bạn (hoặc giữ mặc định):
GITHUB_REPO_URL = "https://github.com/hung187/comfyui-setup1.git" #@param {type:"string"}
# @markdown Điền API Token Civitai của bạn bên dưới:
CIVITAI_TOKEN = "63190c338eed6411b6adbcaecef169bc" #@param {type:"string"}

import os
import subprocess

%cd /content

# 1. Tải hoặc cập nhật bộ script cài đặt từ GitHub
repo_clean = GITHUB_REPO_URL.strip()
setup_dir = "/content/comfyui-setup1"

if repo_clean:
    if not os.path.exists(setup_dir):
        print(f"📦 Tải bộ script setup từ GitHub ({repo_clean})...")
        !git clone "{repo_clean}" "{setup_dir}"
    else:
        print("🔄 Cập nhật bộ script setup từ GitHub...")
        %cd "{setup_dir}"
        !git pull

%cd "{setup_dir}"

# 2. Chạy install.sh cài đặt ComfyUI
token_clean = CIVITAI_TOKEN.strip()
print(f"\n🚀 Bắt đầu chạy script install.sh để chuẩn bị ComfyUI, Models & Nodes...")
!bash install.sh --comfy-dir "/content/ComfyUI" --civitai-token "{token_clean}"

# 3. Tạo Symlink liên kết ComfyUI local với Google Drive
drive_models = "/content/drive/MyDrive/ComfyUI/models"
drive_nodes = "/content/drive/MyDrive/ComfyUI/custom_nodes"
drive_outputs = "/content/drive/MyDrive/ComfyUI_Outputs"

# Đồng bộ models sang Drive
if os.path.exists("/content/ComfyUI/models") and not os.path.islink("/content/ComfyUI/models"):
    !cp -rn /content/ComfyUI/models/* "{drive_models}/" 2>/dev/null || true
    !rm -rf /content/ComfyUI/models
if not os.path.exists("/content/ComfyUI/models"):
    !ln -s "{drive_models}" /content/ComfyUI/models
    print("🔗 Đã liên kết thư mục models với Google Drive!")

# Đồng bộ custom_nodes sang Drive
if os.path.exists("/content/ComfyUI/custom_nodes") and not os.path.islink("/content/ComfyUI/custom_nodes"):
    !cp -rn /content/ComfyUI/custom_nodes/* "{drive_nodes}/" 2>/dev/null || true
    !rm -rf /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes"):
    !ln -s "{drive_nodes}" /content/ComfyUI/custom_nodes
    print("🔗 Đã liên kết thư mục custom_nodes với Google Drive!")

# Đồng bộ output sang Drive
if os.path.exists("/content/ComfyUI/output") and not os.path.islink("/content/ComfyUI/output"):
    !rm -rf /content/ComfyUI/output
if not os.path.exists("/content/ComfyUI/output"):
    !ln -s "{drive_outputs}" /content/ComfyUI/output
    print("🔗 Đã liên kết thư mục output với Google Drive!")

print("\n🎉 ĐÃ HOÀN TẤT ĐỒNG BỘ TOÀN BỘ MODELS VÀ TIỆN ÍCH SANG GOOGLE DRIVE!")


In [ ]:
# @title 📥 (Tùy Chọn) Tải Thêm Checkpoint / LoRA / Custom Node Bất Kỳ Vào Google Drive
# @markdown Nhập thông tin để tải thêm tệp bất kỳ vào Google Drive (để trống nếu không tải thêm):
CIVITAI_TOKEN = "63190c338eed6411b6adbcaecef169bc" #@param {type:"string"}
URL_TAI_THEM = "" #@param {type:"string"}
LOAI_TAI = "checkpoints" #@param ["checkpoints", "loras", "controlnet", "vae", "custom_nodes"]
TEN_FILE_LUU = "" #@param {type:"string"}

import os

if URL_TAI_THEM.strip():
    TOKEN = CIVITAI_TOKEN.strip()
    url = URL_TAI_THEM.strip()
    if "civitai.com" in url and "token=" not in url:
        sep = "&" if "?" in url else "?"
        url += f"{sep}token={TOKEN}"
    
    if LOAI_TAI == "custom_nodes":
        target_dir = f"/content/drive/MyDrive/ComfyUI/custom_nodes"
        os.makedirs(target_dir, exist_ok=True)
        print(f"📦 Đang clone Custom Node vào Drive: {url}")
        !git clone "{url}" "{target_dir}/$(basename '{url}' .git)"
    else:
        target_dir = f"/content/drive/MyDrive/ComfyUI/models/{LOAI_TAI}"
        os.makedirs(target_dir, exist_ok=True)
        out_name = TEN_FILE_LUU.strip() if TEN_FILE_LUU.strip() else os.path.basename(url.split("?")[0])
        out_path = os.path.join(target_dir, out_name)
        print(f"⬇️ Đang tải vào Drive ({target_dir}): {out_name}")
        !aria2c --console-log-level=error --summary-interval=0 -c -x 4 -s 4 -k 1M "{url}" -d "{target_dir}" -o "{out_name}" || \
         wget -c -O "{out_path}" "{url}"
        print(f"✅ Đã tải xong tệp vào Google Drive: {out_path}")
else:
    print("ℹ️ Ô URL trống, bỏ qua tải thêm.")


In [ ]:
# @title 3. Khởi động ComfyUI & Mở Đường Hầm Kết Nối (Cloudflare & Localtunnel)
import os
import time
import subprocess
import re
import urllib.request

# 1. Tải và cài đặt Cloudflared nếu chưa có
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("📦 Đang cài đặt Cloudflared...")
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    !rm -f cloudflared-linux-amd64.deb
    print("✅ Đã cài đặt Cloudflared!")

# Cài nodejs npm cho localtunnel dự phòng
!apt-get update -qq && apt-get install -y -qq nodejs npm > /dev/null 2>&1

%cd /content/ComfyUI

# 2. Khởi chạy ComfyUI Core trực tiếp từ Colab SSD (Model nằm trên Drive via Symlink)
comfy_cmd = "python main.py --listen 0.0.0.0 --port 8188 --enable-cors-header"
comfy_log = "/content/comfyui.log"

if os.path.exists(comfy_log):
    os.remove(comfy_log)

subprocess.Popen(f"{comfy_cmd} > {comfy_log} 2>&1", shell=True)
print("⏳ Đang khởi động ComfyUI Server (vui lòng đợi vài giây cho ComfyUI nạp model).../")

# Polling kiểm tra ComfyUI sẵn sàng tại port 8188
comfy_ready = False
for _ in range(60):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8188/", timeout=2) as response:
            if response.status == 200:
                comfy_ready = True
                break
    except Exception:
        pass
    time.sleep(2)

if comfy_ready:
    print("✅ ComfyUI Server đã sẵn sàng tại 127.0.0.1:8188!")
else:
    print("⚠️ ComfyUI chưa sẵn sàng hoặc gặp lỗi trong quá trình khởi động! Đang kiểm tra log:")
    if os.path.exists(comfy_log):
        with open(comfy_log, "r") as f:
            print(f.read()[-1000:])

# 3. Khởi chạy Cloudflare Tunnel ở background
tunnel_cmd = "cloudflared tunnel --url http://127.0.0.1:8188"
tunnel_log = "/content/cloudflared.log"
if os.path.exists(tunnel_log):
    os.remove(tunnel_log)
subprocess.Popen(f"{tunnel_cmd} > {tunnel_log} 2>&1", shell=True)

# 4. Khởi chạy Localtunnel làm đường hầm dự phòng
lt_log = "/content/localtunnel.log"
if os.path.exists(lt_log):
    os.remove(lt_log)
subprocess.Popen(f"npx localtunnel --port 8188 > {lt_log} 2>&1", shell=True)

# Lấy password giải mã của Localtunnel (IP Public Colab)
try:
    colab_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf-8').strip()
except Exception:
    colab_ip = "N/A"

# 5. Quét log tìm Public URLs
cf_url = None
lt_url = None

for _ in range(25):
    time.sleep(1)
    if not cf_url and os.path.exists(tunnel_log):
        with open(tunnel_log, "r") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
            if m:
                cf_url = m.group(0)
    if not lt_url and os.path.exists(lt_log):
        with open(lt_log, "r") as f:
            m = re.search(r"https://[a-zA-Z0-9-]+\.loca\.lt", f.read())
            if m:
                lt_url = m.group(0)
    if cf_url and lt_url:
        break

print("\n" + "═"*75)
if cf_url:
    print(f"🌐 LINK CHÍNH (Cloudflare Tunnel): {cf_url}")
else:
    print("🌐 LINK CHÍNH (Cloudflare): Đang kết nối, kiểm tra log dưới.")

if lt_url:
    print(f"🔄 LINK DỰ PHÒNG (Localtunnel):     {lt_url}")
    print(f"🔑 Mật khẩu nhập vào Localtunnel:  {colab_ip}")

print("📁 Mọi ảnh sinh ra / ảnh train được lưu riêng tại: Google Drive -> ComfyUI_Outputs")
print("💡 Nếu mạng Viettel/VNPT/FPT bị chặn trycloudflare.com, hãy dùng Link Dự Phòng!")
print("═"*75 + "\n")

# 6. Live stream log ComfyUI
print("📋 STREAM LOG COMFYUI (Đang hoạt động...):")
try:
    with open(comfy_log, "r") as f:
        f.seek(0, 2)
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                time.sleep(0.5)
except KeyboardInterrupt:
    print("\n🛑 Đã dừng ComfyUI.")
